[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/4590_Colabs/blob/main/10_ModelToBMI.ipynb)

# **Set 10 — BMI-1: micro:bit Simulator (Browser-Based)**
**Author:** Nair SS, et al., Electrical Engineering and Computer Science, University of Missouri-Columbia, Columbia, MO, USA

**Goal:** Translate a neural simulation's fear probability into a visual "expression" — more LEDs lit, blinking faster, as fear rises — on the free, browser-based **micro:bit simulator**. This module serves as a Brain-Machine Interface (BMI) capstone, translating digital states into a physical-style expression, entirely in the browser.

> **Note:** This notebook needs only a normal Colab runtime and a web browser. No hardware, no USB connection, no Local Runtime.

---

## **Table of Contents**
* [Variable Key and Roadmap](#variable-key-and-roadmap)
* [H0 — Starter: From Probability to a Visual Pattern](#h0-starter-from-probability-to-a-visual-pattern)
* [H1 — FEAR Mapping Logic (Host Side)](#h1-fear-mapping-logic-host-side)
* [H2 — micro:bit Simulator Code: The Expression Layer](#h2-microbit-simulator-code-the-expression-layer)
* [Reflection: The BMI Synthesis](#reflection-the-bmi-synthesis)
* [Practice / Discussion Questions](#practice-discussion-questions)

---
### A Note on This Notebook
**This completed, filled-in notebook is the lab deliverable.** Add your own cells as you go. **H1 — FEAR Mapping Logic** below is a good spot for an optional (AI-assisted) extension.

---
### 📋 Before You Begin: How This Module Is Graded
Within this module's combined Submissions + Quizzes score, the companion quiz is worth **66.7%** and the Colab submission is worth the remaining **33.3%**. Submissions + Quizzes together (across all 10 modules) make up **30%** of your overall course grade. The quiz specifically checks whether the understanding behind your submission is actually yours — pasting a classmate's code or an AI's explanation can get this notebook looking complete, but it won't get you through a quiz question applying the same concept to a scenario you haven't seen before.

The **design problem** below, and the **two questions you'll create yourself**, both exist for exactly this reason.

---

## **H0 — Starter: From Probability to a Visual Pattern**
*Defining the Probability → Pattern Mapping*

**Idea:** Before we can show an "expression," we need a rule connecting a number (the model's fear probability, 0 to 1) to something visual. The micro:bit's display is a 5×5 grid of LEDs — 25 pixels total. We'll map fear probability to two things: **how many of those 25 LEDs are lit**, and **how fast they blink**. Low fear → a few LEDs, blinking slowly. High fear → most/all LEDs, blinking fast.

#### **Predict → Verify**
* **Prediction:** `fear_to_pattern(0.0)` should return a small lit-count and a slow (large) blink interval. `fear_to_pattern(1.0)` should return a large lit-count and a fast (small) blink interval.
* **Verification:** Run the cell below and confirm the printed values match your prediction at both ends, and move smoothly in between.

In [ ]:
# ===== WHAT THIS CELL DOES =====
# Installs "ipywidgets", a small add-on that lets us put sliders and buttons
# in the notebook (used in H1 below).
# WHAT YOU DO: just run this cell (click the play button, or press Shift+Enter).
# You only need to run it once per session. Nothing to change.

%pip install ipywidgets

In [ ]:
# ===== WHAT THIS CELL DOES =====
# Defines a "function" -- a reusable mini-recipe -- called fear_to_pattern.
# You give it a fear level (a number from 0 to 1) and it hands back TWO numbers:
#   1) lit         = how many of the micro:bit's 25 LEDs should be turned on
#   2) interval_ms = how long (in milliseconds) between blinks
#                    (a SMALLER number means FASTER blinking)
# WHAT YOU DO: just run this cell. Nothing to change yet.

def fear_to_pattern(prob):
    # "prob" is the fear level: 0.0 = completely calm, 1.0 = maximum fear.

    # Safety step: if someone passes in a number outside 0-1, squeeze it back
    # into range. (max(0.0, ...) stops it going below 0; min(1.0, ...) stops it
    # going above 1.)
    prob = max(0.0, min(1.0, prob))

    # How many LEDs to light: starts at 1 when prob=0 (so a calm brain still
    # shows a tiny flicker), and grows to 25 when prob=1. round() makes it a
    # whole number, since you can't light 7.4 LEDs.
    lit = round(1 + prob * 24)

    # Blink spacing: 1000 ms (1 second, slow) when calm, shrinking to 100 ms
    # (very fast) at maximum fear. int() makes it a whole number.
    interval_ms = int(1000 - 900 * prob)

    # Hand the two answers back to whoever called this function.
    return lit, interval_ms

print('H0 ready. Mapping function defined -- no hardware required.')

In [ ]:
# ===== WHAT THIS CELL DOES =====
# Tries out fear_to_pattern on five fear levels so you can SEE the pattern:
# as fear goes up, more LEDs light up and the blink interval shrinks.
# WHAT YOU DO: run it, compare the output to your prediction in the text above.
# TRY THIS: edit the list of numbers in the brackets below and run again.

# "for p in [...]" means: repeat the indented lines once for each number in the list.
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    # Ask our function from H0 for the pattern at this fear level.
    lit, interval_ms = fear_to_pattern(p)
    # Print one summary line. The f'...' lets us drop values into the text:
    # {p:.2f} = p shown with 2 decimals; {lit:2d} = lit as a whole number.
    print(f'prob={p:.2f}  ->  {lit:2d} LEDs lit, blinking every {interval_ms} ms')

Notice the trend: as `prob` rises, **more LEDs light up** and the **blink interval shrinks** (faster blinking). That's the entire "hardware bridge" we need this time — just this one function. Everything from here is about using it.

---

## **H1 — FEAR Mapping Logic (Host Side)**
*From a Fluid Probability to a Live Preview*

**Idea:** Your neural model produces a fluid probability of fear (ranging from 0 to 1). We use `fear_to_pattern()` from H0 directly — no threshold needed for the base display, since we want a **graded** response, not a binary one. We'll also add one optional graded-on-top-of-graded feature: an **alert overlay** that flashes a special icon when fear crosses a threshold, so the threshold/cooldown idea (useful in many real BMI designs) still has a place here.

#### **The Control Levers**
* **`prob` (Fear Probability):** The continuous 0–1 signal from your neural model. Drives the base LED-count/blink-speed pattern directly.
* **`thr` (Alert Threshold):** Where the *optional* alert overlay kicks in on top of the graded pattern. A low threshold (0.3) makes the alert "jumpy"; a high threshold (0.8) makes it reserved for only strong fear states.
* **`cd_ms` (Cooldown):** How long the alert must wait before it can fire again, preventing rapid repeat-triggering from a noisy signal.

#### **Exercises**
1.  **Manual Preview:** Move the **"Fear Probability"** slider and click **"Preview Pattern"** to see the resulting LED count and blink speed printed.
2.  **Sensitivity Sweep:** Try `prob` near 0.1, then near 0.9. Confirm the pattern changes smoothly, not in jumps.
3.  **Alert Threshold:** Set the alert threshold to 0.7 and sweep `prob` past it. Notice the alert only fires once per cooldown window, even if `prob` stays above threshold.

**Make It Your Own (optional AI-assisted extension).** The alert overlay above is binary (fires / doesn't fire). Extend it to **3 discrete alert levels** (e.g., calm/alert/critical) instead of one threshold, and update the micro:bit simulator code in H2 to show a different icon at each level.

> **Suggested prompt:** *"Here is my fear probability -> LED pattern mapping: [paste your H0/H1 code]. I want 3 discrete alert levels instead of one threshold. Show me what to change here and in the micro:bit simulator code below."*

*Your Answer Here (what you changed, and how you tested it):*

In [ ]:
# ===== WHAT THIS CELL DOES =====
# Builds a small control panel: a slider (your pretend "fear level") and a
# button. Click the button and the notebook prints what the micro:bit WOULD show
# at that fear level -- no hardware needed. It also demonstrates the optional
# "alert" idea: if fear crosses a threshold, an ALERT message fires, but then
# stays quiet for a short "cooldown" so it doesn't repeat endlessly.
# WHAT YOU DO: run the cell, drag the slider, click "Preview Pattern", and repeat.
# TRY THIS: change threshold=0.7 or cd_ms=1500 in the function below, re-run,
# and see how the alert behavior changes.

import time                      # lets us read the clock (for the cooldown timer)
import ipywidgets as w           # the slider/button toolkit installed earlier
from IPython.display import display   # shows the widgets in the notebook

# Create the slider (value goes from 0 to 1) and the button.
prob_slider = w.FloatSlider(value=0.3, min=0, max=1, step=0.01, description='Fear prob')
b = w.Button(description='Preview Pattern', button_style='danger')

# A little "memory" that remembers WHEN the alert last fired (0 = never).
_alert_state = {'t0': 0}

def on_preview(prob, threshold=0.7, cd_ms=1500):
    # threshold = fear level at which the alert should fire
    # cd_ms     = cooldown: minimum milliseconds between two alerts

    # Step 1: use our H0 function to get the graded pattern for this fear level.
    lit, interval_ms = fear_to_pattern(prob)
    print(f'Graded pattern: {lit} LEDs lit, blinking every {interval_ms} ms (prob={prob:.2f})')

    # Step 2: check the clock (time.time() is in seconds; *1000 converts to ms).
    now = time.time() * 1000

    # Step 3: fire the alert only if BOTH are true:
    #   (a) fear is at or above the threshold, AND
    #   (b) enough time has passed since the last alert (the cooldown).
    if prob >= threshold and (now - _alert_state['t0']) > cd_ms:
        _alert_state['t0'] = now   # remember when this alert happened
        print(f'ALERT overlay triggered! (prob={prob:.2f} crossed threshold={threshold})')

def _preview_click(_):
    # This runs every time the button is clicked: it reads the slider's current
    # value and passes it to on_preview.
    on_preview(prob_slider.value)

# Connect the button to the function above, then show slider + button together.
b.on_click(_preview_click)
display(w.VBox([prob_slider, b]))

## **H2 — micro:bit Simulator Code: The Expression Layer**
*Run in the Browser Simulator — No Flashing, No Hardware*

**Idea:** This is the device-side code that actually draws the pattern: it sweeps through a demo sequence of fear probabilities, computing lit-count and blink-speed from the exact same rule as H0/H1, and animates the 5×5 LED grid accordingly — low fear shows a few LEDs blinking slowly, high fear shows most/all LEDs blinking rapidly.

#### **How to Run It (automated)**
1.  Run the code cell below. It will **print the code**, then do two things automatically for you: **copy the code to your clipboard**, and **open the simulator in a new browser tab**.
2.  Switch to that new tab — it should be [python.microbit.org](https://python.microbit.org/v/3).
3.  Click inside the code editor, select all the starter code (Ctrl+A / Cmd+A), and **paste** (Ctrl+V / Cmd+V) to replace it with your copied code.
4.  Click the simulator's own **Run/Play** button (top of the on-screen micro:bit) — no connecting, no flashing, no USB. Watch the LED grid respond.

#### **If the automation doesn't work (fallback)**
Some browsers block pop-ups or clipboard access by default — this is normal and not a bug. If the tab doesn't open or the paste comes up empty:
* **No tab opened?** Your browser likely blocked the pop-up. Open [python.microbit.org](https://python.microbit.org/v/3) yourself in a new tab.
* **Clipboard paste is empty?** Scroll up to this cell's printed output, select the code by hand, and copy it (Ctrl+C / Cmd+C) before pasting into the editor.

#### **Exercises**
1.  **Custom Expression:** Change the alert icon from `Image.SKULL` to `Image.GHOST` or `Image.SAD`.
2.  **Audio Profile:** The current alert tone is a descending "alarm." Can you change the `music.play` list to a high-pitched "startle" sound instead?

In [ ]:
# ===== WHAT THIS CELL DOES =====
# 1) Holds a short program (written in MicroPython, the micro:bit's language)
#    inside a text block called microbit_code. That program draws the LED
#    pattern from H0/H1 on the simulator: few slow-blinking LEDs when calm,
#    many fast-blinking LEDs when afraid.
# 2) Prints that program so you can read it.
# 3) Tries to COPY it to your clipboard and OPEN the simulator in a new browser tab.
#
# WHAT YOU DO:
#   a) Run this cell.
#   b) Switch to the new tab that opens (python.microbit.org).
#   c) Click in the code editor, select all (Ctrl+A / Cmd+A), paste (Ctrl+V / Cmd+V).
#   d) Click the simulator's Run/Play button.
# If no tab opens or the paste is empty, see the fallback steps in the text above.
# You do NOT need to understand or edit the JavaScript at the bottom -- it just
# automates the copy-and-open steps.

# --- PART 1: the micro:bit program, stored as plain text between the triple quotes ---
microbit_code = '''
from microbit import *   # load the micro:bit's built-in tools (display, sleep, etc.)
import music             # load the sound tools

def pattern_for(prob):
    # Same rule as H0/H1 (copied here because the simulator is a separate
    # program and can't see our Colab functions).
    lit = round(1 + prob * 24)       # how many LEDs to light (1 to 25)
    interval = int(1000 - 900 * prob)  # blink spacing in ms (1000 = slow, 100 = fast)
    return lit, interval

def show_pattern(lit):
    # Turn on exactly "lit" LEDs, filling the 5x5 grid row by row.
    display.clear()                  # start with all LEDs off
    count = 0                        # how many LEDs we've turned on so far
    for y in range(5):               # go through the 5 rows...
        for x in range(5):           # ...and the 5 columns in each row
            if count < lit:          # still need more LEDs on?
                display.set_pixel(x, y, 9)   # turn this LED on (9 = full brightness)
                count += 1           # tally it

def alert():
    # Optional "alarm" for very high fear: show a skull and play 3 descending notes.
    display.show(Image.SKULL)
    music.play(['c5:1', 'b4:1', 'a4:2'])

# A made-up fear history: calm -> terrified -> calm again. In a real BMI this
# list would come from your neural model's output.
demo_probs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 0.8, 0.6, 0.4, 0.2, 0.0]

while True:                          # repeat forever (until you press Stop)
    for prob in demo_probs:          # step through each fear level in turn
        lit, interval = pattern_for(prob)   # how many LEDs, how fast?
        if prob >= 0.9:              # extremely high fear...
            alert()                  # ...trigger the alarm
        show_pattern(lit)            # draw the LEDs
        sleep(interval)              # wait (LEDs stay ON for this long)
        display.clear()              # turn all LEDs off
        sleep(interval)              # wait (LEDs stay OFF for this long) -> blinking!
'''

# --- PART 2: show the program in this cell's output ---
print('Generated micro:bit code (see below). Attempting to copy it to your clipboard')
print('and open the simulator in a new tab now -- check the instructions above if either')
print('one does not happen automatically.')
print()
print(microbit_code)

# --- PART 3: convenience automation (you do not need to edit or understand this) ---
# This small piece of JavaScript runs in YOUR web browser (not on Google's
# servers). It copies the program above to your clipboard and opens the
# simulator website in a new tab. It CANNOT type into the simulator for you --
# browsers don't allow one website to control another, which is why you still
# paste manually.
import json as _json
from IPython.display import Javascript, display

# Convert our program text into a form JavaScript can safely hold.
_code_json = _json.dumps(microbit_code)

# The JavaScript itself. "__CODE_JSON__" is a placeholder swapped in below.
_js = """
(function() {
  const code = __CODE_JSON__;
  try {
    // Copy the program to the clipboard (browsers may ask permission or refuse).
    navigator.clipboard.writeText(code).then(function() {
      console.log('micro:bit code copied to clipboard.');
    }).catch(function(err) {
      console.warn('Clipboard copy failed (normal in some browsers):', err);
    });
  } catch (e) {
    console.warn('Clipboard API unavailable:', e);
  }
  // Open the simulator in a new tab (pop-up blockers may stop this).
  window.open('https://python.microbit.org/v/3', '_blank');
})();
""".replace("__CODE_JSON__", _code_json)

# Run the JavaScript in your browser.
display(Javascript(_js))

### 🎯 Design Problem — Apply Threshold Tuning
**Scenario:** Your alert overlay (H1) uses a decision threshold on top of the continuous fear-probability signal, to decide when to flash the special alert icon. The probability signal has some baseline noise.

**If the threshold is set too low, what's the likely failure mode? Too high?**

A) Too low → false triggers from background noise; Too high → real signals get missed entirely
B) Too low → real signals get missed; Too high → false triggers from noise
C) Threshold level has no effect on triggering behavior, only on timing
D) Both too-low and too-high thresholds produce identical behavior

*Reason it out before revealing the answer.*

**Answer: A).**

A threshold set too low means ordinary background noise can cross it, triggering false alert activations. A threshold set too high means the real signal may never reliably cross it, causing missed alerts. This is the exact same trade-off from Set 4's BCI data bridge (`data_offset` tuning) — now applied to the alert overlay on top of a graded, simulator-driven display instead of a simulated signal alone.

**Note:** this module doesn't yet have a formal categorized Knowledge Bank the way earlier Sets do — that's still on the to-do list. This design problem and the create-your-own activity below are built directly from the existing H0–H2 content in the meantime.

### 📸 Upload Checkpoint 1 — The Threshold-Tuning Reasoning, From Memory
**Before you write anything: stop, close this notebook (or at minimum, don't scroll back up).** Without looking back, redo this reasoning from memory — same scenario (threshold too low vs. too high), same two failure modes — from scratch.

Only after you've written what you can recall should you check it against your own earlier work in this notebook. Then photograph your hand-written page and upload it below.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file (`from google.colab import files`) and display it inline.

In [ ]:
# ===== YOUR TURN =====
# Write (or ask an AI assistant to help you write) a few lines of code that:
#   1) lets you upload a photo of your hand-written page, and
#   2) shows the photo right here in the notebook.
# Hint for your AI prompt: mention "Google Colab", "files.upload()", and
# "display the uploaded image inline".
# Your image upload + display code here:


### 🎯 Your Turn — Design Two Questions, Not Just Answer Them
Pick any concept from this Set. Create **two** short questions of your own — one of each type:

**1. An Analysis-type question:** present a scenario, plot, or result, and ask the reader to interpret *why* something happened or what it implies.

**2. A Design-type question:** ask the reader *how* to achieve a specific target outcome, or what change would produce a given effect.

For each: include a brief setup, 2–4 possible answers, the correct answer, and a short explanation of why. No required format or topic beyond that — design both however makes sense to you. If your questions would genuinely catch someone who only skimmed this Set, you've demonstrated real understanding.

*Your Analysis-type question, answers, and explanation here:*

*Your Design-type question, answers, and explanation here:*

### 📸 Upload Checkpoint 2 — Sketch the Full Signal Pipeline
Without looking back through this notebook, sketch the complete pipeline from memory: the neural model's internal state, through the probability -> pattern mapping (H0/H1), into the generated MicroPython code (H2), to the simulated LED display in the browser simulator. Label each stage and what crosses between them.

**Activity — vibe-code it:** ask an AI assistant to help you write a short Colab cell that lets you upload an image file and display it inline.

In [ ]:
# ===== YOUR TURN =====
# Write (or ask an AI assistant to help you write) a few lines of code that:
#   1) lets you upload a photo of your hand-written page, and
#   2) shows the photo right here in the notebook.
# Hint for your AI prompt: mention "Google Colab", "files.upload()", and
# "display the uploaded image inline".
# Your image upload + display code here:


## **Practice / Discussion Questions — Set 10**
*Translation to Brain-Machine Interfaces*

1.  **System Design:** Define a specific *internal model state* you would export to a prosthetic device (real or simulated) and justify why it is a meaningful control signal.
2.  **Thresholding:** Why is a simple threshold rule often preferred over complex ML pipelines for real-time educational BMIs?
3.  **Timing Mismatch:** *Predict:* What happens if the internal neural events occur at a much faster frequency than the display (or any real effector) can visibly update? How do you fix this mismatch?
4.  **Error Modes:** Identify one *software* error mode specific to this pipeline (e.g., an off-by-one in the LED-counting loop, a rounding error that causes flicker right at a pattern-count boundary, an interval value hitting zero at extreme probabilities) and propose a mitigation.
5.  **Encoding Graded States:** This module already encodes graded fear as LED count + blink speed. Propose a *different* graded encoding (e.g., per-pixel brightness 0–9, a sound-pitch sweep) and discuss the trade-offs versus the one used here.
6.  **Debouncing:** Why is it important to "smooth" or debounce the neural signal before it reaches the mapping logic in H0/H1?
7.  **Jitter Sources:** List two sources of *timing jitter* that could make the simulator's visual response feel inconsistent even if the underlying model is running perfectly.
8.  **Verification:** Design a simple unit test for `fear_to_pattern()` — which probability values would you test (don't forget the edges, `prob=0` and `prob=1`), and what output do you expect at each?
9.  **Scaling:** If you added a second simulated effector (e.g., a rising-pitch tone playing alongside the LED pattern), how would you structure the code so both effectors stay synchronized to the same `prob`?
10. **Mechanistic Connection:** How does Set 10 reinforce the idea that behavior is an emergent property of underlying mechanistic circuit logic?